## Ý tưởng chính của additive smoothing

Trong language model, một vấn đề rất hay gặp là có những n-gram chưa từng xuất hiện trong tập train. Nếu tính xác suất kiểu đếm thô thông thường, các n-gram đó sẽ có xác suất bằng 0.

Điều này rất bất tiện vì chỉ cần một thành phần có xác suất bằng 0 là xác suất của cả chuỗi có thể rơi về 0 luôn.

Additive smoothing xử lý chuyện đó bằng cách cộng thêm một hằng số nhỏ `alpha` vào số lần đếm:
- nếu `alpha = 1` thì gọi là **Laplace smoothing**,
- nếu `0 < alpha < 1` thì thường gọi là **Lidstone smoothing**.

Nói dễ hiểu: ta bớt tự tin hơn với những gì đã thấy và chừa ra một ít xác suất cho những trường hợp chưa thấy.

## Cell khởi tạo dữ liệu ví dụ

Bài trên web dùng một corpus rất nhỏ gồm 2 câu:
- `the cat sat on the mat`
- `the dog sat on the mat`

Mình tạo riêng cell này để ta có cùng dữ liệu gốc cho toàn bộ phần sau.

In [1]:
corpus = [
    'the cat sat on the mat',
    'the dog sat on the mat'
]

corpus

['the cat sat on the mat', 'the dog sat on the mat']

## Xem nhanh unigram, bigram và vocabulary

Cell này không bắt buộc trong bài gốc, nhưng mình thêm vào để nhìn trực quan hơn.

Khi hiểu được các số đếm ban đầu, bạn sẽ thấy công thức smoothing bớt trừu tượng hơn rất nhiều.

In [2]:
from collections import defaultdict

unigrams = defaultdict(int)
bigrams = defaultdict(int)
vocab = set()

for sentence in corpus:
    tokens = sentence.split()
    vocab.update(tokens)
    for i, token in enumerate(tokens):
        unigrams[token] += 1
        if i > 0:
            bigrams[(tokens[i - 1], tokens[i])] += 1

print('Vocabulary:', sorted(vocab))
print('Vocab size:', len(vocab))
print('Count(the):', unigrams['the'])
print("Count(('the', 'cat')):", bigrams[('the', 'cat')])
print("Count(('the', 'dog')):", bigrams[('the', 'dog')])
print("Count(('the', 'sat')):", bigrams[('the', 'sat')])

Vocabulary: ['cat', 'dog', 'mat', 'on', 'sat', 'the']
Vocab size: 6
Count(the): 4
Count(('the', 'cat')): 1
Count(('the', 'dog')): 1
Count(('the', 'sat')): 0


## So sánh với cách tính không smoothing

Trước khi vào Laplace và Lidstone, mình tính thử xác suất bigram theo kiểu đếm thô:

`P(word2 | word1) = count(word1, word2) / count(word1)`

Mục đích là để thấy ngay vấn đề: nếu một bigram chưa từng xuất hiện thì xác suất bằng 0.

In [3]:
def unsmoothed_bigram_prob(word1, word2, bigrams, unigrams):
    if unigrams[word1] == 0:
        return 0.0
    return bigrams[(word1, word2)] / unigrams[word1]

print("P(cat | the) without smoothing:", round(unsmoothed_bigram_prob('the', 'cat', bigrams, unigrams), 3))
print("P(dog | the) without smoothing:", round(unsmoothed_bigram_prob('the', 'dog', bigrams, unigrams), 3))
print("P(sat | the) without smoothing:", round(unsmoothed_bigram_prob('the', 'sat', bigrams, unigrams), 3))

P(cat | the) without smoothing: 0.25
P(dog | the) without smoothing: 0.25
P(sat | the) without smoothing: 0.0


## Laplace smoothing

Đây là phần code chính đầu tiên trong bài GeeksforGeeks.

Với Laplace smoothing, ta cộng `1` vào mọi bigram count. Công thức trong bài là:

`P(w_n | w_{n-1}) = (C(w_{n-1}, w_n) + 1) / (C(w_{n-1}) + V)`

Trong đó `V` là kích thước từ vựng. Cách làm này giúp cả những bigram chưa từng thấy cũng có xác suất lớn hơn 0.

In [4]:
from collections import defaultdict

class LaplaceSmoothing:
    def __init__(self, corpus):
        self.unigrams = defaultdict(int)
        self.bigrams = defaultdict(int)
        self.total_unigrams = 0
        self.vocab_size = 0

        self.train(corpus)

    def train(self, corpus):
        vocab = set()
        for sentence in corpus:
            tokens = sentence.split()
            vocab.update(tokens)
            for i in range(len(tokens)):
                self.unigrams[tokens[i]] += 1
                self.total_unigrams += 1
                if i > 0:
                    self.bigrams[(tokens[i - 1], tokens[i])] += 1

        self.vocab_size = len(vocab)

    def bigram_prob(self, word1, word2):
        return (self.bigrams[(word1, word2)] + 1) / (self.unigrams[word1] + self.vocab_size)

model = LaplaceSmoothing(corpus)
print(f"P(cat | the): {model.bigram_prob('the', 'cat'):.3f}")

P(cat | the): 0.200


## Đọc kết quả Laplace cho dễ hiểu hơn

Trong corpus này:
- `count(the, cat) = 1`
- `count(the) = 4`
- `V = 6`

Nên:

`P(cat | the) = (1 + 1) / (4 + 6) = 2/10 = 0.2`

Cell dưới đây in ra thêm vài xác suất để thấy rõ Laplace đã kéo xác suất của bigram chưa thấy từ `0` lên thành một giá trị nhỏ dương.

In [5]:
print(f"P(cat | the) with Laplace: {model.bigram_prob('the', 'cat'):.3f}")
print(f"P(dog | the) with Laplace: {model.bigram_prob('the', 'dog'):.3f}")
print(f"P(sat | the) with Laplace: {model.bigram_prob('the', 'sat'):.3f}")

P(cat | the) with Laplace: 0.200
P(dog | the) with Laplace: 0.200
P(sat | the) with Laplace: 0.100


## Lidstone smoothing

Đây là phần code chính thứ hai của bài.

Lidstone smoothing tổng quát hơn Laplace: thay vì cộng `1`, ta cộng `alpha` với `0 < alpha < 1`.

Công thức:

`P(w_n | w_{n-1}) = (C(w_{n-1}, w_n) + alpha) / (C(w_{n-1}) + alpha * V)`

Điểm hay là mình điều chỉnh được độ smoothing. Nếu cộng quá mạnh như Laplace, đôi khi xác suất của các bigram xuất hiện thường xuyên lại bị kéo xuống hơi nhiều.

In [6]:
class LidstoneSmoothing(LaplaceSmoothing):
    def __init__(self, corpus, lambda_=0.5):
        self.lambda_ = lambda_
        super().__init__(corpus)

    def bigram_prob(self, word1, word2):
        return (self.bigrams[(word1, word2)] + self.lambda_) / (self.unigrams[word1] + self.lambda_ * self.vocab_size)

model = LidstoneSmoothing(corpus, lambda_=0.5)
print(f"P(cat | the): {model.bigram_prob('the', 'cat'):.3f}")

P(cat | the): 0.214


## So sánh trực tiếp: no smoothing vs Laplace vs Lidstone

Cell này là phần mình thêm vào để bạn nhìn một lượt là hiểu.

Ta sẽ so sánh ba trường hợp cho cùng một prefix là `the`:
- bigram đã thấy như `the -> cat`,
- bigram đã thấy như `the -> dog`,
- bigram chưa thấy như `the -> sat`.

Đây là chỗ đẹp nhất để thấy smoothing thật sự làm gì.

In [7]:
laplace_model = LaplaceSmoothing(corpus)
lidstone_model = LidstoneSmoothing(corpus, lambda_=0.5)

candidates = ['cat', 'dog', 'sat']
for word in candidates:
    p_raw = unsmoothed_bigram_prob('the', word, bigrams, unigrams)
    p_laplace = laplace_model.bigram_prob('the', word)
    p_lidstone = lidstone_model.bigram_prob('the', word)
    print(f"word = {word:>3} | raw = {p_raw:.3f} | laplace = {p_laplace:.3f} | lidstone = {p_lidstone:.3f}")

word = cat | raw = 0.250 | laplace = 0.200 | lidstone = 0.214
word = dog | raw = 0.250 | laplace = 0.200 | lidstone = 0.214
word = sat | raw = 0.000 | laplace = 0.100 | lidstone = 0.071


## Giải thích kết quả so sánh

Khi nhìn kết quả ở cell trên, bạn sẽ thấy:
- với bigram chưa từng thấy như `the -> sat`, cách tính thô cho ra `0`, còn smoothing cho ra số dương,
- với bigram đã thấy, smoothing sẽ làm xác suất giảm xuống một chút vì mô hình phải chia bớt xác suất cho các trường hợp chưa thấy,
- Lidstone thường "dịu" hơn Laplace nếu chọn `alpha < 1`, nên xác suất của các bigram đã thấy thường bị kéo xuống ít hơn.

Đó chính là lý do Lidstone hay thực tế hơn Laplace trong nhiều bài toán.

## Thử nhiều giá trị alpha khác nhau

Cell cuối cùng này cũng là phần mình thêm để notebook dễ học hơn.

Ta thử `alpha = 1.0`, `0.5`, `0.1` để thấy rằng:
- `alpha` càng lớn thì smoothing càng mạnh,
- `alpha` càng nhỏ thì mô hình càng gần với cách đếm gốc hơn,
- nhưng nếu quá nhỏ thì unseen n-gram vẫn được cứu, chỉ là được cứu ít hơn.

In [8]:
for alpha in [1.0, 0.5, 0.1]:
    temp_model = LidstoneSmoothing(corpus, lambda_=alpha)
    print(f"alpha = {alpha:.1f} -> P(cat | the) = {temp_model.bigram_prob('the', 'cat'):.3f}, P(sat | the) = {temp_model.bigram_prob('the', 'sat'):.3f}")

alpha = 1.0 -> P(cat | the) = 0.200, P(sat | the) = 0.100
alpha = 0.5 -> P(cat | the) = 0.214, P(sat | the) = 0.071
alpha = 0.1 -> P(cat | the) = 0.239, P(sat | the) = 0.022
